In [ ]:
import pandas as pd
import json

print("Setup Successful")

In [ ]:
import json

with open("candidate_schema.json", "r") as f:
    schema = json.load(f)

print(schema)

In [ ]:
count = 0

with open("candidates.jsonl", "r") as f:
    for line in f:
        count += 1

print("Total Candidates:", count)

In [ ]:
import json

with open("candidates.jsonl", "r") as f:
    first_candidate = json.loads(next(f))

print(first_candidate)

In [ ]:
import json
from pprint import pprint

with open("candidates.jsonl", "r") as f:
    first_candidate = json.loads(next(f))

pprint(first_candidate)

In [ ]:
import json

with open("candidates.jsonl", "r") as f:
    first_candidate = json.loads(next(f))

print(first_candidate.keys())

In [ ]:
for key in first_candidate.keys():
    print(key)

In [ ]:
for key, value in first_candidate.items():
    print("\n====================")
    print(key)
    print(type(value))

In [ ]:
print(first_candidate.keys())

In [ ]:
print(first_candidate["redrob_signals"])

In [ ]:
print(first_candidate["profile"])

In [ ]:
!pip install pandas numpy sentence-transformers scikit-learn python-docx tqdm

In [ ]:
from docx import Document

doc = Document("job_description.docx")

jd_text = ""

for para in doc.paragraphs:
    jd_text += para.text + "\n"

print(jd_text[:5000])

In [ ]:
import json

candidate_texts = []
candidate_ids = []

with open("candidates.jsonl", "r") as f:

    for line in f:

        candidate = json.loads(line)

        profile = candidate.get("profile", {})
        skills = candidate.get("skills", [])
        career = candidate.get("career_history", [])

        text = ""

        text += profile.get("headline", "") + " "
        text += profile.get("summary", "") + " "

        for skill in skills:
            text += skill.get("name", "") + " "

        for job in career:
            text += job.get("title", "") + " "
            text += job.get("description", "") + " "

        candidate_texts.append(text)

        candidate_ids.append(candidate["candidate_id"])

print("Total Candidates:", len(candidate_texts))
print(candidate_texts[0][:1000])

In [ ]:
jd_keywords = [
    "embeddings",
    "retrieval",
    "ranking",
    "llm",
    "fine-tuning",
    "vector database",
    "milvus",
    "pinecone",
    "weaviate",
    "qdrant",
    "faiss",
    "elasticsearch",
    "opensearch",
    "sentence-transformers",
    "python",
    "ndcg",
    "mrr",
    "map",
    "a/b testing",
    "evaluation"
]

print(jd_keywords)

In [ ]:
def keyword_score(text):

    text = text.lower()

    score = 0

    for keyword in jd_keywords:
        if keyword.lower() in text:
            score += 1

    return score

In [ ]:
score = keyword_score(candidate_texts[0])

print("Score:", score)

In [ ]:
scores = []

for i in range(100):

    score = keyword_score(candidate_texts[i])

    scores.append(
        (candidate_ids[i], score)
    )

scores = sorted(
    scores,
    key=lambda x: x[1],
    reverse=True
)

print(scores[:10])

In [ ]:
lengths = [len(text) for text in candidate_texts]

print("Average Length:", sum(lengths)/len(lengths))
print("Maximum Length:", max(lengths))
print("Minimum Length:", min(lengths))

In [ ]:
def extract_candidate_features(candidate):

    profile = candidate.get("profile", {})
    signals = candidate.get("redrob_signals", {})

    years_exp = profile.get("years_of_experience", 0)

    recruiter_score = signals.get(
        "recruiter_response_rate",
        0
    )

    github_score = signals.get(
        "github_activity_score",
        0
    )

    interview_score = signals.get(
        "interview_completion_rate",
        0
    )

    open_to_work = int(
        signals.get(
            "open_to_work_flag",
            False
        )
    )

    return {
        "years_exp": years_exp,
        "recruiter_score": recruiter_score,
        "github_score": github_score,
        "interview_score": interview_score,
        "open_to_work": open_to_work
    }

In [ ]:
import json

with open("candidates.jsonl", "r") as f:
    first_candidate = json.loads(next(f))

print(
    extract_candidate_features(
        first_candidate
    )
)

In [ ]:
def recruiter_score(candidate_text, features):

    score = 0

    # Keyword Match (40 points)

    jd_keywords = [
        "embedding",
        "retrieval",
        "ranking",
        "llm",
        "python",
        "vector",
        "milvus",
        "pinecone",
        "evaluation",
        "search"
    ]

    keyword_hits = 0

    text = candidate_text.lower()

    for word in jd_keywords:
        if word in text:
            keyword_hits += 1

    score += (keyword_hits / len(jd_keywords)) * 40


    # Experience (20 points)

    exp = features["years_exp"]

    if 5 <= exp <= 9:
        score += 20
    elif exp >= 3:
        score += 10


    # Github Activity (10 points)

    score += min(features["github_score"], 10)


    # Interview Completion (15 points)

    score += features["interview_score"] * 15


    # Recruiter Response (10 points)

    score += features["recruiter_score"] * 10


    # Open To Work (5 points)

    if features["open_to_work"]:
        score += 5

    return round(score, 2)

In [ ]:
features = extract_candidate_features(first_candidate)

score = recruiter_score(
    candidate_texts[0],
    features
)

print(score)

In [ ]:
top_candidates = []

for i in range(1000):

    candidate = json.loads(open("candidates.jsonl").readlines()[i])

    features = extract_candidate_features(candidate)

    score = recruiter_score(
        candidate_texts[i],
        features
    )

    top_candidates.append(
        (candidate_ids[i], score)
    )

print("Finished")

In [ ]:
top_candidates = sorted(
    top_candidates,
    key=lambda x: x[1],
    reverse=True
)

top_candidates[:20]

In [ ]:
import json

top_candidates = []

with open("candidates.jsonl", "r") as f:

    for i, line in enumerate(f):

        if i >= 10000:
            break

        candidate = json.loads(line)

        features = extract_candidate_features(candidate)

        score = recruiter_score(
            candidate_texts[i],
            features
        )

        top_candidates.append(
            (candidate["candidate_id"], score)
        )

print("Finished!")

In [ ]:
top_candidates = sorted(
    top_candidates,
    key=lambda x: x[1],
    reverse=True
)

print(top_candidates[:20])

In [ ]:
import pandas as pd

submission = pd.DataFrame(
    top_candidates,
    columns=["candidate_id", "score"]
)

submission.head()

In [ ]:
submission.to_csv(
    "ranked_candidates.csv",
    index=False
)

print("CSV Created")

In [ ]:
len(top_candidates)

In [ ]:
print("candidate_texts:", len(candidate_texts))
print("candidate_ids:", len(candidate_ids))

In [ ]:
submission.shape

In [ ]:
top_candidates[:5]

In [ ]:
top_candidates[-5:]

In [ ]:
top_candidates = []

In [ ]:
for i in range(1000):
    candidate = json.loads(open("candidates.jsonl").readlines()[i])

    features = extract_candidate_features(candidate)

    score = recruiter_score(
        candidate_texts[i],
        features
    )

    top_candidates.append(
        (candidate_ids[i], score)
    )

print(len(top_candidates))

In [ ]:
print(candidate_ids[123])
print(candidate_ids[124])
print(candidate_ids[125])

In [ ]:
top_candidates = sorted(
    top_candidates,
    key=lambda x: x[1],
    reverse=True
)

print(top_candidates[:10])

In [ ]:
submission = pd.DataFrame(
    top_candidates,
    columns=["candidate_id", "score"]
)

submission.head()

In [ ]:
submission.shape

In [ ]:
len(top_candidates)

In [ ]:
top_candidates = []

In [ ]:
with open("candidates.jsonl", "r") as f:

    for i, line in enumerate(f):

        candidate = json.loads(line)

        features = extract_candidate_features(candidate)

        score = recruiter_score(
            candidate_texts[i],
            features
        )

        top_candidates.append(
            (candidate["candidate_id"], score)
        )

print(len(top_candidates))

In [ ]:
print(len(top_candidates))

In [ ]:
submission = pd.DataFrame(
    top_candidates,
    columns=["candidate_id", "score"]
)

print(submission.shape)

In [ ]:
top_candidates = sorted(
    top_candidates,
    key=lambda x: x[1],
    reverse=True
)

In [ ]:
submission = pd.DataFrame(
    top_candidates,
    columns=["candidate_id", "score"]
)

In [ ]:
submission.head()

In [ ]:
submission.to_csv(
    "ranked_candidates.csv",
    index=False
)

print("Final CSV Created")

In [ ]:
submission.shape

In [ ]:
submission.head()

In [ ]:
submission.tail()

In [ ]:
submission.head(10)

In [ ]:
submission["candidate_id"].nunique()

In [ ]:
submission.isnull().sum()

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
jd_embedding = model.encode(jd_text)

In [ ]:
candidate_embedding = model.encode(candidate_texts[0])

similarity = cosine_similarity(
    [jd_embedding],
    [candidate_embedding]
)[0][0]

print(similarity)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def hybrid_score(candidate_text, features):

    candidate_embedding = model.encode(candidate_text)

    similarity = cosine_similarity(
        [jd_embedding],
        [candidate_embedding]
    )[0][0]

    semantic_score = similarity * 30

    rule_score = recruiter_score(
        candidate_text,
        features
    )

    return round(rule_score + semantic_score, 2)

In [ ]:
features = extract_candidate_features(first_candidate)

score = hybrid_score(
    candidate_texts[0],
    features
)

print(score)

In [ ]:
hybrid_candidates = []

with open("candidates.jsonl", "r") as f:

    for i, line in enumerate(f):

        if i >= 100:
            break

        candidate = json.loads(line)

        features = extract_candidate_features(candidate)

        score = hybrid_score(
            candidate_texts[i],
            features
        )

        hybrid_candidates.append(
            (candidate["candidate_id"], score)
        )

print(len(hybrid_candidates))

In [ ]:
hybrid_candidates = sorted(
    hybrid_candidates,
    key=lambda x: x[1],
    reverse=True
)

In [ ]:
hybrid_candidates[:10]

In [ ]:
import json

top_candidates = []

with open("candidates.jsonl", "r") as f:

    for i, line in enumerate(f):

        if i >= 100000:
            break

        candidate = json.loads(line)

        features = extract_candidate_features(candidate)

        score = hybrid_score(
            candidate_texts[i],
            features
        )

        top_candidates.append(
            (candidate["candidate_id"], score)
        )

        if i % 1000 == 0:
            print("Processed:", i)

print("Finished")
print("Total Candidates:", len(top_candidates))

In [ ]:
print(len(top_candidates))

In [ ]:
submission = pd.DataFrame(
    top_candidates,
    columns=["candidate_id", "score"]
)

print(submission.shape)

In [ ]:
submission = submission.sort_values(
    by="score",
    ascending=False
)

In [ ]:
submission.to_csv(
    "ranked_candidates_hybrid.csv",
    index=False
)

print("CSV Created")

In [ ]:
submission.head()

In [ ]:
submission.isnull().sum()

In [ ]:
jd_skills = [
    "python",
    "embeddings",
    "retrieval",
    "ranking",
    "llm",
    "fine-tuning",
    "milvus",
    "pinecone",
    "weaviate",
    "qdrant",
    "faiss",
    "elasticsearch",
    "opensearch",
    "evaluation",
    "ndcg",
    "mrr",
    "map"
]

print(len(jd_skills))

In [ ]:
def skill_match_score(candidate_text):

    text = candidate_text.lower()

    matches = 0

    for skill in jd_skills:

        if skill in text:
            matches += 1

    score = (matches / len(jd_skills)) * 100

    return round(score, 2)

In [ ]:
print(
    skill_match_score(
        candidate_texts[0]
    )
)

In [ ]:
def final_hybrid_score(
    semantic_score,
    recruiter_score,
    skill_score
):

    return (
        0.5 * semantic_score +
        0.3 * recruiter_score +
        0.2 * skill_score
    )

In [ ]:
hybrid_candidates = []